# QUANTITATIVE PORTFOLIO CORRELATION & DIVERSIFICATION ANALYZER (QPCDA)
## Equity Return Correlation & Lowest Correlation Pair Selection (VN-Index 2023 - 2026)
**Tickers Analyzed:** VCB, BID, CTG, HPG, GVR, DCM, VNM, SAB, MSN, GAS, POW, REE
**Methodology:** Pearson Correlation, Spearman Rank Correlation, 60D/120D Rolling Dynamics, and Sub-period Robustness.

In [ ]:
# Dependencies setup for Google Colab / Jupyter environment
!pip install yfinance openpyxl seaborn scipy matplotlib pandas numpy

In [ ]:
"""
QUANTITATIVE STOCK CORRELATION ANALYSIS & LOWEST CORRELATION PAIR SELECTION
===========================================================================
Model Name: Quantitative Portfolio Correlation & Diversification Analyzer (QPCDA)
Author: Quantitative Finance & Portfolio Management Specialist
Description: Production pipeline for analyzing correlation structure among 12 Vietnamese stocks
             (VCB, BID, CTG, HPG, GVR, DCM, VNM, SAB, MSN, GAS, POW, REE)
             from 2023-07-31 to 2026-07-31, identifying the lowest correlation pair,
             performing robustness checks (Spearman, Rolling 60D/120D, Sub-periods),
             generating visualizations and exporting comprehensive Excel/PNG reports.
"""

import os
import sys
import logging
import warnings

# Ensure UTF-8 output on Windows console
if sys.platform.startswith('win'):
    try:
        sys.stdout.reconfigure(encoding='utf-8')
        sys.stderr.reconfigure(encoding='utf-8')
    except AttributeError:
        pass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import yfinance as yf

# Configure warnings and plotting style
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial', 'Helvetica']
plt.rcParams['axes.unicode_minus'] = False

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger("QPCDA_Model")

# ==============================================================================
# 1. MODEL CONFIGURATION & METADATA
# ==============================================================================
MODEL_NAME = "Quantitative Portfolio Correlation & Diversification Analyzer (QPCDA)"
OUTPUT_DIR = os.path.join(os.getcwd(), 'output')
os.makedirs(OUTPUT_DIR, exist_ok=True)

START_DATE = '2023-07-31'
END_DATE = '2026-08-01'  # Inclusive of trading day 2026-07-31

TICKERS_INFO = {
    'VCB': {'Company': 'Vietcombank', 'Sector': 'Banking'},
    'BID': {'Company': 'BIDV', 'Sector': 'Banking'},
    'CTG': {'Company': 'VietinBank', 'Sector': 'Banking'},
    'HPG': {'Company': 'Hoa Phat Group', 'Sector': 'Basic Materials / Steel'},
    'GVR': {'Company': 'Vietnam Rubber Group', 'Sector': 'Industrials / Chemicals'},
    'DCM': {'Company': 'PetroVietnam Ca Mau Fertilizer', 'Sector': 'Chemicals / Fertilizers'},
    'VNM': {'Company': 'Vinamilk', 'Sector': 'Consumer Staples / Dairy'},
    'SAB': {'Company': 'Sabeco', 'Sector': 'Consumer Staples / Beverages'},
    'MSN': {'Company': 'Masan Group', 'Sector': 'Consumer Staples / Conglomerate'},
    'GAS': {'Company': 'PV GAS', 'Sector': 'Energy / Utilities / Natural Gas'},
    'POW': {'Company': 'PV Power', 'Sector': 'Utilities / Electric Power'},
    'REE': {'Company': 'Refrigeration Electrical Engineering', 'Sector': 'Utilities / Infrastructure'}
}
TICKER_LIST = list(TICKERS_INFO.keys())

# Sub-periods definition for stability inspection
SUB_PERIODS = [
    ('Sub-period 1 (2023-2024)', '2023-07-31', '2024-07-31'),
    ('Sub-period 2 (2024-2025)', '2024-07-31', '2025-07-31'),
    ('Sub-period 3 (2025-2026)', '2025-07-31', '2026-07-31')
]

# ==============================================================================
# 2. DATA COLLECTION ENGINE
# ==============================================================================
def collect_stock_data(tickers, start_date, end_date):
    """
    Downloads historical Adjusted Closing Price data for Vietnamese equities from Yahoo Finance.
    """
    logger.info(f"[{MODEL_NAME}] Step 1: Downloading adjusted historical prices...")
    yf_tickers = [f"{t}.VN" for t in tickers]
    ticker_map = {f"{t}.VN": t for t in tickers}
    
    try:
        raw_data = yf.download(
            yf_tickers,
            start=start_date,
            end=end_date,
            auto_adjust=True,  # Automatically calculates and returns dividend/split-adjusted close
            progress=True
        )
        
        if raw_data.empty:
            raise ValueError("Downloaded dataset from Yahoo Finance is empty.")
            
        if isinstance(raw_data.columns, pd.MultiIndex):
            adj_close = raw_data['Close'].copy()
        else:
            adj_close = raw_data[['Close']].copy()
            
        adj_close.columns = [ticker_map.get(col, col.replace('.VN', '')) for col in adj_close.columns]
        adj_close = adj_close[tickers]  # Preserve specified ordering
        
        logger.info(f"Successfully downloaded price series for {len(tickers)} stocks ({start_date} to {end_date}).")
        return adj_close
        
    except Exception as e:
        logger.error(f"Error encountered during data acquisition: {e}")
        raise

# ==============================================================================
# 3. DATA CLEANING & AUDITING ENGINE
# ==============================================================================
def clean_and_audit_data(df_prices):
    """
    Cleans market data: filters out non-trading sessions, handles duplicates, and audits quality.
    """
    logger.info(f"[{MODEL_NAME}] Step 2: Cleaning and auditing market observations...")
    
    # 1. Drop dates where all assets are NaN (national holidays, weekends)
    df_cleaned = df_prices.dropna(how='all').copy()
    
    # 2. Check and remove duplicate dates
    dup_dates = df_cleaned.index[df_cleaned.index.duplicated()].tolist()
    if dup_dates:
        logger.warning(f"Detected {len(dup_dates)} duplicated dates. Keeping first occurrence.")
        df_cleaned = df_cleaned[~df_cleaned.index.duplicated(keep='first')]
        
    total_sessions = len(df_cleaned)
    null_counts = df_cleaned.isnull().sum()
    valid_counts = df_cleaned.notnull().sum()
    pct_missing = (null_counts / total_sessions) * 100
    
    quality_audit = pd.DataFrame({
        'Ticker': TICKER_LIST,
        'Company Name': [TICKERS_INFO[t]['Company'] for t in TICKER_LIST],
        'Sector': [TICKERS_INFO[t]['Sector'] for t in TICKER_LIST],
        'Total Sessions': total_sessions,
        'Valid Observations': [valid_counts[t] for t in TICKER_LIST],
        'Missing Sessions': [null_counts[t] for t in TICKER_LIST],
        'Missing Ratio (%)': [round(pct_missing[t], 2) for t in TICKER_LIST]
    })
    
    common_sample_sessions = len(df_cleaned.dropna())
    logger.info(f"Total valid market trading sessions: {total_sessions}.")
    logger.info(f"Common sample sessions (complete across all 12 stocks): {common_sample_sessions}.")
    
    return df_cleaned, quality_audit

# ==============================================================================
# 4. RETURN CALCULATION ENGINE
# ==============================================================================
def compute_returns(df_prices):
    """
    Computes Daily Simple Returns (R_t = P_t / P_{t-1} - 1)
    and Daily Logarithmic Returns (r_t = ln(P_t / P_{t-1})).
    """
    logger.info(f"[{MODEL_NAME}] Step 3: Computing daily simple and log return series...")
    simple_returns = df_prices.pct_change()
    log_returns = np.log(df_prices / df_prices.shift(1))
    return simple_returns, log_returns

# ==============================================================================
# 5. CORRELATION MATRIX & PAIR RANKING ENGINE
# ==============================================================================
def analyze_correlations(returns_df):
    """
    Constructs 12x12 Pearson & Spearman Correlation Matrices,
    audits pairwise observations, and ranks all 66 asset combinations.
    """
    logger.info(f"[{MODEL_NAME}] Step 4: Constructing correlation matrices and ranking 66 asset pairs...")
    
    # 1. Pearson Correlation Matrix (Pairwise Complete Observations)
    pearson_matrix = returns_df.corr(method='pearson')
    
    # 2. Spearman Rank Correlation Matrix
    spearman_matrix = returns_df.corr(method='spearman')
    
    # 3. Observation counts matrix
    tickers = returns_df.columns.tolist()
    n = len(tickers)
    obs_matrix = pd.DataFrame(index=tickers, columns=tickers, dtype=int)
    pairs_list = []
    
    for i in range(n):
        for j in range(n):
            t1, t2 = tickers[i], tickers[j]
            valid_pair = returns_df[[t1, t2]].dropna()
            n_obs = len(valid_pair)
            obs_matrix.loc[t1, t2] = n_obs
            
            # Upper triangle elements (N = 12 * 11 / 2 = 66 unique pairs)
            if i < j:
                p_corr = valid_pair[t1].corr(valid_pair[t2], method='pearson')
                s_corr = valid_pair[t1].corr(valid_pair[t2], method='spearman')
                pairs_list.append({
                    'Stock 1': t1,
                    'Stock 2': t2,
                    'Pair Name': f"{t1} - {t2}",
                    'Sector 1': TICKERS_INFO[t1]['Sector'],
                    'Sector 2': TICKERS_INFO[t2]['Sector'],
                    'Pearson Correlation': p_corr,
                    'Spearman Correlation': s_corr,
                    'Observations': n_obs
                })
                
    df_pairs = pd.DataFrame(pairs_list)
    df_pairs = df_pairs.sort_values(by='Pearson Correlation', ascending=True).reset_index(drop=True)
    df_pairs['Rank'] = df_pairs.index + 1
    
    cols_order = ['Rank', 'Stock 1', 'Stock 2', 'Pair Name', 'Pearson Correlation', 
                  'Spearman Correlation', 'Observations', 'Sector 1', 'Sector 2']
    df_pairs = df_pairs[cols_order]
    
    top5_lowest = df_pairs.head(5).copy()
    lowest_pair_info = df_pairs.iloc[0]
    
    logger.info(f"Lowest Correlation Pair Identified: {lowest_pair_info['Pair Name']} (Pearson r = {lowest_pair_info['Pearson Correlation']:.4f})")
    
    # Audit negative correlation pairs
    neg_pairs = df_pairs[df_pairs['Pearson Correlation'] < 0]
    if len(neg_pairs) > 0:
        most_neg = neg_pairs.iloc[0]
        logger.info(f"Most Negative Pair: {most_neg['Pair Name']} (Pearson r = {most_neg['Pearson Correlation']:.4f})")
    else:
        logger.info("No negative correlation pairs found over the full 3-year sample.")
        
    return pearson_matrix, spearman_matrix, obs_matrix, df_pairs, top5_lowest, lowest_pair_info

# ==============================================================================
# 6. STABILITY & ROBUSTNESS AUDIT ENGINE
# ==============================================================================
def audit_stability(returns_df, stock1, stock2):
    """
    Computes rolling correlation (60-day and 120-day windows)
    and breaks down performance across three distinct annual sub-periods.
    """
    logger.info(f"[{MODEL_NAME}] Step 5: Auditing correlation stability for {stock1} vs {stock2}...")
    pair_returns = returns_df[[stock1, stock2]].dropna()
    
    rolling_60 = pair_returns[stock1].rolling(window=60).corr(pair_returns[stock2])
    rolling_120 = pair_returns[stock1].rolling(window=120).corr(pair_returns[stock2])
    
    df_rolling = pd.DataFrame({
        f'{stock1}_Return': pair_returns[stock1],
        f'{stock2}_Return': pair_returns[stock2],
        'Rolling_60D': rolling_60,
        'Rolling_120D': rolling_120
    })
    
    sub_results = []
    for name, s_date, e_date in SUB_PERIODS:
        sub_slice = pair_returns.loc[s_date:e_date]
        if len(sub_slice) > 10:
            p_corr = sub_slice[stock1].corr(sub_slice[stock2], method='pearson')
            s_corr = sub_slice[stock1].corr(sub_slice[stock2], method='spearman')
            sub_results.append({
                'Sub-period': name,
                'Start Date': s_date,
                'End Date': e_date,
                'Observations': len(sub_slice),
                'Pearson Correlation': p_corr,
                'Spearman Correlation': s_corr
            })
        else:
            sub_results.append({
                'Sub-period': name,
                'Start Date': s_date,
                'End Date': e_date,
                'Observations': len(sub_slice),
                'Pearson Correlation': np.nan,
                'Spearman Correlation': np.nan
            })
            
    df_subperiods = pd.DataFrame(sub_results)
    return df_rolling, df_subperiods

# ==============================================================================
# 7. QUANTITATIVE VISUALIZATION ENGINE
# ==============================================================================
def generate_visualizations(returns_df, pearson_mat, top5_df, lowest_pair_info, df_rolling):
    """
    Generates high-resolution visualization suites:
    7.1 12x12 Pearson Correlation Heatmap
    7.2 Top 5 Lowest Correlation Pairs Horizontal Bar Chart
    7.3 Linear Regression Scatter Plot with OLS equation and R^2
    7.4 Rolling Correlation Dynamics (60-day & 120-day)
    """
    logger.info(f"[{MODEL_NAME}] Step 6: Rendering quantitative visualization suite...")
    stock1 = lowest_pair_info['Stock 1']
    stock2 = lowest_pair_info['Stock 2']
    
    fig, axes = plt.subplots(2, 2, figsize=(20, 16), dpi=300)
    
    # -------------------------------------------------------------
    # 7.1 Pearson Correlation Heatmap
    # -------------------------------------------------------------
    ax1 = axes[0, 0]
    sns.heatmap(
        pearson_mat,
        annot=True,
        fmt=".2f",
        cmap="coolwarm",
        vmin=-0.1,
        vmax=1.0,
        center=0.0,
        linewidths=0.5,
        cbar_kws={'label': 'Pearson Correlation Coefficient (ρ)'},
        ax=ax1,
        square=True
    )
    ax1.set_title("7.1. Pearson Correlation Matrix (12 VN-Index Equities)", fontsize=14, weight='bold', pad=12)
    ax1.set_xlabel("Ticker", fontsize=11, weight='bold')
    ax1.set_ylabel("Ticker", fontsize=11, weight='bold')

    # -------------------------------------------------------------
    # 7.2 Top 5 Lowest Correlation Pairs Bar Chart
    # -------------------------------------------------------------
    ax2 = axes[0, 1]
    pairs_labels = top5_df['Pair Name'].tolist()[::-1]
    corrs = top5_df['Pearson Correlation'].tolist()[::-1]
    colors = ['#2ca02c' if c < 0.20 else '#1f77b4' for c in corrs]
    
    bars = ax2.barh(pairs_labels, corrs, color=colors, height=0.55, edgecolor='black', alpha=0.85)
    ax2.set_xlim(0, max(corrs) * 1.35)
    ax2.set_xlabel("Pearson Correlation Coefficient (ρ)", fontsize=11, weight='bold')
    ax2.set_title("7.2. Top 5 Lowest Correlation Pairs (Diversification Ranking)", fontsize=14, weight='bold', pad=12)
    
    for bar in bars:
        width = bar.get_width()
        ax2.text(width + 0.005, bar.get_y() + bar.get_height()/2, f"{width:.4f}",
                 va='center', ha='left', fontsize=11, weight='bold', color='navy')
    ax2.grid(True, linestyle='--', alpha=0.6, axis='x')

    # -------------------------------------------------------------
    # 7.3 Return Scatter Plot & Linear Regression
    # -------------------------------------------------------------
    ax3 = axes[1, 0]
    valid_pair = returns_df[[stock1, stock2]].dropna()
    x = valid_pair[stock1]
    y = valid_pair[stock2]
    
    slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
    r_squared = r_value**2
    
    ax3.scatter(x, y, alpha=0.45, color='#1f77b4', edgecolors='none', s=28, label='Daily Return Pairs')
    x_line = np.linspace(x.min(), x.max(), 100)
    y_line = slope * x_line + intercept
    ax3.plot(x_line, y_line, color='crimson', linewidth=2, 
             label=f'OLS Fit: y = {slope:.3f}x + {intercept:.4f}')
    
    stats_text = (
        f"Pair: {stock1} - {stock2}\n"
        f"Pearson r = {r_value:.4f}\n"
        f"R² = {r_squared:.4f}\n"
        f"p-value = {p_value:.4e}\n"
        f"Observations = {len(valid_pair)}"
    )
    ax3.text(0.05, 0.95, stats_text, transform=ax3.transAxes, verticalalignment='top',
             bbox=dict(boxstyle='round,pad=0.6', facecolor='white', edgecolor='gray', alpha=0.9),
             fontsize=10, fontweight='bold')
    ax3.axhline(0, color='black', linestyle=':', alpha=0.5)
    ax3.axvline(0, color='black', linestyle=':', alpha=0.5)
    ax3.set_xlabel(f"Daily Simple Return: {stock1} ({TICKERS_INFO[stock1]['Company']})", fontsize=11, weight='bold')
    ax3.set_ylabel(f"Daily Simple Return: {stock2} ({TICKERS_INFO[stock2]['Company']})", fontsize=11, weight='bold')
    ax3.set_title(f"7.3. Return Scatter Plot & OLS Fit ({stock1} vs {stock2})", fontsize=14, weight='bold', pad=12)
    ax3.legend(loc='lower right', frameon=True)
    ax3.grid(True, linestyle='--', alpha=0.6)

    # -------------------------------------------------------------
    # 7.4 Rolling Correlation Dynamics
    # -------------------------------------------------------------
    ax4 = axes[1, 1]
    ax4.plot(df_rolling.index, df_rolling['Rolling_60D'], label='Rolling 60 Trading Days', color='#e66101', linewidth=1.8)
    ax4.plot(df_rolling.index, df_rolling['Rolling_120D'], label='Rolling 120 Trading Days', color='#5e3c99', linewidth=2.2)
    ax4.axhline(0, color='black', linestyle='-', alpha=0.7, linewidth=1)
    ax4.axhline(lowest_pair_info['Pearson Correlation'], color='crimson', linestyle='--', alpha=0.8,
                label=f"Full-sample Pearson r ({lowest_pair_info['Pearson Correlation']:.4f})")
    ax4.set_xlabel("Date", fontsize=11, weight='bold')
    ax4.set_ylabel("Pearson Correlation Coefficient (ρ)", fontsize=11, weight='bold')
    ax4.set_title(f"7.4. Rolling Correlation Dynamics ({stock1} - {stock2})", fontsize=14, weight='bold', pad=12)
    ax4.legend(loc='upper right', frameon=True)
    ax4.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()
    chart_path = os.path.join(OUTPUT_DIR, 'Correlation_Charts.png')
    plt.savefig(chart_path, dpi=300, bbox_inches='tight')
    plt.close()
    logger.info(f"Saved combined master chart at: {chart_path}")
    
    # Save individual charts for modular reporting
    # 1. Heatmap
    plt.figure(figsize=(10, 8), dpi=300)
    sns.heatmap(pearson_mat, annot=True, fmt=".2f", cmap="coolwarm", vmin=-0.1, vmax=1.0, center=0.0,
                linewidths=0.5, cbar_kws={'label': 'Pearson Correlation Coefficient (ρ)'}, square=True)
    plt.title("Pearson Correlation Matrix (12 VN-Index Equities)", fontsize=13, weight='bold', pad=12)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, '01_Correlation_Heatmap.png'), dpi=300, bbox_inches='tight')
    plt.close()
    
    # 2. Top 5 Lowest Pairs
    plt.figure(figsize=(9, 5), dpi=300)
    bars = plt.barh(pairs_labels, corrs, color=colors, height=0.5, edgecolor='black', alpha=0.85)
    plt.xlim(0, max(corrs) * 1.35)
    plt.xlabel("Pearson Correlation Coefficient (ρ)", fontsize=11, weight='bold')
    plt.title("Top 5 Lowest Correlation Pairs (Diversification Ranking)", fontsize=13, weight='bold', pad=12)
    for bar in bars:
        w = bar.get_width()
        plt.text(w + 0.005, bar.get_y() + bar.get_height()/2, f"{w:.4f}", va='center', ha='left', fontsize=11, weight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, '02_Top5_Lowest_Pairs.png'), dpi=300, bbox_inches='tight')
    plt.close()
    
    # 3. Scatter Plot
    plt.figure(figsize=(8, 6), dpi=300)
    plt.scatter(x, y, alpha=0.45, color='#1f77b4', edgecolors='none', s=30, label='Daily Return Pairs')
    plt.plot(x_line, y_line, color='crimson', linewidth=2, label=f'OLS Fit: y = {slope:.3f}x + {intercept:.4f}')
    plt.text(0.05, 0.95, stats_text, transform=plt.gca().transAxes, verticalalignment='top',
             bbox=dict(boxstyle='round,pad=0.6', facecolor='white', edgecolor='gray', alpha=0.9),
             fontsize=10, fontweight='bold')
    plt.axhline(0, color='black', linestyle=':', alpha=0.5)
    plt.axvline(0, color='black', linestyle=':', alpha=0.5)
    plt.xlabel(f"Daily Simple Return: {stock1} ({TICKERS_INFO[stock1]['Company']})", fontsize=11, weight='bold')
    plt.ylabel(f"Daily Simple Return: {stock2} ({TICKERS_INFO[stock2]['Company']})", fontsize=11, weight='bold')
    plt.title(f"Return Scatter Plot & OLS Fit ({stock1} vs {stock2})", fontsize=13, weight='bold', pad=12)
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, '03_Return_Scatter_Plot.png'), dpi=300, bbox_inches='tight')
    plt.close()
    
    # 4. Rolling Correlation
    plt.figure(figsize=(10, 5), dpi=300)
    plt.plot(df_rolling.index, df_rolling['Rolling_60D'], label='Rolling 60 Trading Days', color='#e66101', linewidth=1.8)
    plt.plot(df_rolling.index, df_rolling['Rolling_120D'], label='Rolling 120 Trading Days', color='#5e3c99', linewidth=2.2)
    plt.axhline(0, color='black', linestyle='-', alpha=0.7, linewidth=1)
    plt.axhline(lowest_pair_info['Pearson Correlation'], color='crimson', linestyle='--', alpha=0.8,
                label=f"Full-sample Pearson r ({lowest_pair_info['Pearson Correlation']:.4f})")
    plt.xlabel("Date", fontsize=11, weight='bold')
    plt.ylabel("Pearson Correlation Coefficient (ρ)", fontsize=11, weight='bold')
    plt.title(f"Rolling Correlation Dynamics ({stock1} - {stock2})", fontsize=13, weight='bold', pad=12)
    plt.legend(loc='upper right')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, '04_Rolling_Correlation.png'), dpi=300, bbox_inches='tight')
    plt.close()

# ==============================================================================
# 8. EXCEL REPORT EXPORT ENGINE
# ==============================================================================
def export_results_to_excel(df_prices, simple_returns, log_returns, quality_audit,
                            pearson_mat, spearman_mat, obs_mat, df_pairs, top5_df, df_subperiods):
    """
    Exports structured, multi-worksheet Excel workbooks in English.
    """
    logger.info(f"[{MODEL_NAME}] Step 7: Exporting results to Excel workbooks...")
    
    # File 1: Stock_Price_Data.xlsx
    file1_path = os.path.join(OUTPUT_DIR, 'Stock_Price_Data.xlsx')
    with pd.ExcelWriter(file1_path, engine='openpyxl') as writer:
        df_prices.to_excel(writer, sheet_name='Adjusted_Closing_Price')
        simple_returns.to_excel(writer, sheet_name='Daily_Simple_Return')
        log_returns.to_excel(writer, sheet_name='Daily_Log_Return')
        quality_audit.to_excel(writer, sheet_name='Data_Quality_Audit', index=False)
        
    # File 2: Correlation_Analysis.xlsx
    file2_path = os.path.join(OUTPUT_DIR, 'Correlation_Analysis.xlsx')
    with pd.ExcelWriter(file2_path, engine='openpyxl') as writer:
        pearson_mat.to_excel(writer, sheet_name='Pearson_Correlation_Matrix')
        spearman_mat.to_excel(writer, sheet_name='Spearman_Correlation_Matrix')
        df_pairs.to_excel(writer, sheet_name='Top_66_Correlation_Pairs', index=False)
        top5_df.to_excel(writer, sheet_name='Top_5_Lowest_Pairs', index=False)
        obs_mat.to_excel(writer, sheet_name='Pairwise_Observation_Counts')
        df_subperiods.to_excel(writer, sheet_name='Sub_period_Results', index=False)
        
    logger.info(f"Successfully generated {file1_path} and {file2_path}.")

# ==============================================================================
# 9. MASTER PIPELINE EXECUTION
# ==============================================================================
def main():
    print("=" * 85)
    print(f"MODEL: {MODEL_NAME.upper()}")
    print("=" * 85)
    
    # Step 1: Collect Data
    df_prices = collect_stock_data(TICKER_LIST, START_DATE, END_DATE)
    
    # Step 2: Clean & Audit Data
    df_cleaned, quality_audit = clean_and_audit_data(df_prices)
    
    # Step 3: Returns Calculation
    simple_returns, log_returns = compute_returns(df_cleaned)
    
    # Step 4: Correlation Matrix & Pair Ranking
    pearson_mat, spearman_mat, obs_mat, df_pairs, top5_df, lowest_pair_info = analyze_correlations(simple_returns)
    
    # Step 5: Stability & Sub-period Audit
    stock1 = lowest_pair_info['Stock 1']
    stock2 = lowest_pair_info['Stock 2']
    df_rolling, df_subperiods = audit_stability(simple_returns, stock1, stock2)
    
    # Step 6: Render Visualizations
    generate_visualizations(simple_returns, pearson_mat, top5_df, lowest_pair_info, df_rolling)
    
    # Step 7: Export to Excel Workbooks
    export_results_to_excel(df_cleaned, simple_returns, log_returns, quality_audit,
                            pearson_mat, spearman_mat, obs_mat, df_pairs, top5_df, df_subperiods)
    
    print("\n" + "=" * 85)
    print("QUANTITATIVE FINANCE & PORTFOLIO SUMMARY")
    print("=" * 85)
    print(f"Model Name                 : {MODEL_NAME}")
    print(f"Number of Stocks Analyzed  : {len(TICKER_LIST)}")
    print(f"Time Horizon               : {START_DATE} to 2026-07-31")
    print(f"Total Asset Pairs Evaluated: {len(df_pairs)}")
    print(f"Primary Metric             : Pearson Correlation (Pairwise Complete Observations)")
    print(f"Lowest Correlation Pair    : [{stock1} - {stock2}] ({TICKERS_INFO[stock1]['Company']} - {TICKERS_INFO[stock2]['Company']})")
    print(f"Pearson Correlation (r)    : {lowest_pair_info['Pearson Correlation']:.4f}")
    print(f"Spearman Correlation (rs)  : {lowest_pair_info['Spearman Correlation']:.4f}")
    print(f"Valid Trading Observations : {lowest_pair_info['Observations']} sessions")
    print("\nTop 5 Lowest Correlation Pairs:")
    print(top5_df[['Rank', 'Pair Name', 'Pearson Correlation', 'Spearman Correlation', 'Observations']].to_string(index=False))
    print("\nSub-period Correlation Audit for Lowest Pair:")
    print(df_subperiods.to_string(index=False))
    print("=" * 85)
    print(f"All reports and visual charts have been saved to: {OUTPUT_DIR}")

if __name__ == '__main__':
    main()
